In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

The types `literal`, `clause`, and `cnf` are defined in the notebook `04-CNF.ipynb`:
```
type literal =
  | Pos of string    (* p *)
  | Neg of string    (* ¬p *)

type clause = literal RSet.t
type cnf    = clause RSet.t
```

In [ ]:
import_notebook "04-CNF.ipynb"

The function `first` takes one argument:
- `s` is a nonempty set.

It returns an element of `s` in a non-random way, namely the smallest element.

In [ ]:
let first (s : 'a RSet.t) : 'a = RSet.choose s

# The <a href="https://en.wikipedia.org/wiki/DPLL_algorithm">Davis-Putnam Algorithm</a> with the Jeroslow-Wang Heuristic

This notebook implements the algorithm of Davis and Putnam.  The variables for a case distinction are selected according to the *Jeroslow-Wang heuristic*.
Further details about this algorithm are provided in the lecture notes.

The function `complement` takes one argument:
- `l` is a literal.

It computes the complement of `l`.  If $p$ is a propositional variable, we have the following:
* $\texttt{complement}(p) = \neg p$,
* $\texttt{complement}(\neg p) = p$.

As we are working with clauses that result from transforming given formulas into *conjunctive normal form* and these clauses do not contain $\top$ or $\bot$, we don't have to bother with $\top$ or $\bot$ in this function.

In [ ]:
let complement (l : literal) : literal =
  match l with
  | Pos p -> Neg p
  | Neg p -> Pos p

The function `extract_variable` takes one argument:
- `l` is a literal.

It returns the variable of `l`.  If $p$ is a propositional variable, we have the following:
* $\texttt{extract\_variable}(p) = p$,
* $\texttt{extract\_variable}(\neg p) = p$.

In [ ]:
let extract_variable (l : literal) : string =
  match l with
  | Pos p | Neg p -> p

The function `select_literal` takes three arguments:
- `clauses` is a set of clauses,
- `variables` is the set of all variables,
- `used_vars` is the set of variables that have already been used in case distinctions.

It returns a literal whose variable is an element of `variables` but not of `used_vars`.  The literal is chosen according to the *Jeroslow-Wang heuristic*.  The Jeroslow-Wang score $\texttt{JW}(\texttt{clauses}, l)$ of a literal $l$ in a set of clauses is defined as follows:
$$
\texttt{JW}(\texttt{clauses}, l) = \sum\limits_{\bigl\{\,C \in \texttt{clauses}\;\mid\; l \in C\;\bigr\}} \frac{1}{\;2^{|C|}\;}
$$
Here, $|C|$ denotes the number of literals in the clause $C$.  The idea is to choose a literal that occurs in many clauses and therefore subsumes a lot of clauses.  Clauses with fewer literals are much more important than clauses with many literals, as it is much easier to satisfy a clause with many literals.  If a clause has $n$ different literals, then the probability that the clause is `false` is only $\displaystyle\frac{1}{2^n}$ if the propositional variables are assigned randomly.  The function returns the literal with the highest score.

**Implementation:** The TypeScript version computes the score of every literal with a separate pass over all clauses.  As the Sudoku in the notebook `10-Sudoku.ipynb` has 729 variables, this makes the algorithm very slow: With separate passes, solving this Sudoku takes about 15 minutes, with a single pass it takes less than a minute.  Therefore, we compute the scores of all literals in a single pass over the clauses:
1. `scores` is a *hash table* that maps literals to their scores.  A hash table is a mutable map.  `Hashtbl.find_opt scores l` returns `Some s` if `l` has the score `s`, and `None` if `l` has no score yet.  `Hashtbl.replace scores l s` sets the score of `l` to `s`.
2. For every clause `c`, the contribution $1/2^{|c|}$ is added to the scores of all literals of `c`.  In <samp>OCaml</samp>, the arithmetic operators for floating point numbers carry a dot: `+.`, `/.`, and `**` computes a power.  `float_of_int` converts an integer into a floating point number.
3. The variables that have not been used are processed in ascending order.  For every such variable $p$, the literals $p$ and $\neg p$ are considered.  A literal that does not occur in any clause has the score 0.  The accumulator of `RSet.fold` and `List.fold_left` is the pair `(best, best_score)` consisting of the best literal found so far and its score.  Initially, the score is `neg_infinity`, i.e. $-\infty$, so that the first literal that is considered replaces the initial value of `best`.  As a literal replaces the current best literal only if its score is strictly greater, the first literal with the highest score is returned.  This is the same literal that the TypeScript version selects.

In [ ]:
let select_literal (clauses : cnf) (variables : string RSet.t)
                   (used_vars : string RSet.t) : literal =
  let scores : (literal, float) Hashtbl.t = Hashtbl.create 1024 in
  let score l = Option.value (Hashtbl.find_opt scores l) ~default:0. in
  RSet.iter
    (fun c ->
       let weight = 1. /. 2. ** float_of_int (RSet.cardinal c) in
       RSet.iter (fun l -> Hashtbl.replace scores l (score l +. weight)) c)
    clauses;
  let consider (best, best_score) l =
    if score l > best_score then (l, score l) else (best, best_score)
  in
  let (best, _) =
    RSet.fold
      (fun p acc -> List.fold_left consider acc [Pos p; Neg p])
      (RSet.diff variables used_vars)
      (Pos (first variables), neg_infinity)
  in
  best

The function `reduce` takes two arguments:
- `clauses` is a set of clauses,
- `l` is a literal.

It performs all unit cuts and all unit subsumptions on clauses of the set `clauses` that are possible using the unit clause $\{\mathtt{l}\}$.  It returns a pair `(reduced, new_units)`:
- `reduced` is the resulting set of clauses.
- `new_units` is the set of unit clauses that are created by the unit cuts.  These unit clauses have not been used for reductions yet.

Mathematically, the set `reduced` is defined as follows:
$$\texttt{reduce}(\texttt{clauses},l)  :=
 \Bigl\{\, C \backslash \bigl\{\overline{\,l\,}\bigr\} \;|\; C \in \texttt{clauses} \wedge \overline{\,l\,} \in C \,\Bigr\}
       \,\cup\, \Bigl\{\, C \in \texttt{clauses} \mid \overline{\,l\,} \not\in C \wedge l \not\in C \Bigr\} \cup \bigl\{\{l\}\bigr\}.
$$
This function should only be called if the unit clause $\{l\}$ is an element of the set `clauses`.

**Implementation:** Only the clauses that contain $l$ or $\overline{\,l\,}$ are affected by the unit clause $\{l\}$.  They are collected in the set `affected`, which is usually much smaller than `clauses`.
- The affected clauses that contain $l$ are dropped (unit subsumption).
- From the other affected clauses, which contain $\overline{\,l\,}$, the literal $\overline{\,l\,}$ is removed (unit cut).  `RSet.filter_map` computes the set `shortened` of these clauses.
- All clauses that are not affected remain unchanged.  They form the set `RSet.diff clauses affected`.

A unit cut with the unit clause $\{l\}$ creates the unit clause $\{k\}$ from every clause of the form $\{\overline{\,l\,}, k\}$ where $k \neq l$.  As all other clauses either remain unchanged or are removed, the unit clauses in `shortened` are exactly the unit clauses that are created by the unit cuts.

The TypeScript version applies `filterMap` to all clauses.  For the module `RSet`, this would be slow:  A set is a sorted list, and removing $\overline{\,l\,}$ from a clause changes the position of the clause in this order.  Therefore, the result of `RSet.filter_map` has to be sorted again.  Here, only the few shortened clauses have to be sorted.  `RSet.filter` and `RSet.diff` keep the order of the clauses, and `RSet.union` merges the unchanged clauses with the shortened clauses.  This makes the solution of the Sudoku in the notebook `10-Sudoku.ipynb` almost twice as fast.

In [ ]:
let reduce (clauses : cnf) (l : literal) : cnf * cnf =
  let l_bar     = complement l in
  let affected  = RSet.filter (fun c -> RSet.mem l c || RSet.mem l_bar c) clauses in
  let shortened =
    RSet.filter_map
      (fun c -> if RSet.mem l c then None else Some (RSet.remove l_bar c))
      affected
  in
  let new_units = RSet.filter (fun c -> RSet.cardinal c = 1) shortened in
  let reduced   = RSet.union (RSet.diff clauses affected) shortened in
  (RSet.add (RSet.singleton l) reduced, new_units)

The function `saturate` takes two arguments:
- `clauses` is a set of clauses,
- `units` is a set of unit clauses that have not yet been used for unit cuts and unit subsumptions.  Every unit clause $\{l\}$ of `clauses` that is not an element of `units` must already have been used, i.e. neither $l$ nor $\overline{\,l\,}$ occurs in any other clause of `clauses`.

It computes the set of those clauses that can be derived from $\texttt{clauses} \cup \texttt{units}$ via repeated applications of unit cuts or unit subsumptions.

**Implementation:** The recursive function `loop` takes two arguments: `s` is the current set of clauses and `pending` contains the unit clauses of `s` that have not yet been used.  As long as there is a unit clause that has not been used, we use it to reduce `s`.  Then we remove it from `pending` and add the unit clauses that are created by this reduction.  As soon as `s` contains the empty clause, `s` is unsatisfiable and we stop, since further reductions are pointless.  The pattern `let (reduced, new_units) = ...` splits the pair returned by `reduce` into its components.

Two observations make this efficient:
- We never have to search `s` for new unit clauses, since `reduce` returns them.
- A unit clause has to be used only once:  After the reduction with the unit clause $\{l\}$, neither $l$ nor $\overline{\,l\,}$ occurs in any other clause of `s`.  As later reductions only remove or shorten clauses, this remains true.

In [ ]:
let saturate (clauses : cnf) (units : cnf) : cnf =
  let rec loop (s : cnf) (pending : cnf) : cnf =
    if RSet.is_empty pending || RSet.mem RSet.empty s then
      s
    else
      let unit_clause = first pending in
      let (reduced, new_units) = reduce s (first unit_clause) in
      loop reduced (RSet.union (RSet.remove unit_clause pending) new_units)
  in
  loop (RSet.union clauses units) units

The function `solve` takes one argument:
- `clauses` is a set of clauses.

The function tries to compute a variable assignment that satisfies all clauses in `clauses`.  If this is successful, a set of unit clauses is returned.  This set of unit clauses does not contain any complementary literals and therefore corresponds to a variable assignment satisfying all clauses.  If the set `clauses` is unsatisfiable, then the set `{{}}` is returned instead.

The real work is done by the function `solve_recursive`.  This function takes three additional arguments:
* `units` is a set of unit clauses that have not yet been used for unit cuts and unit subsumptions.  Every unit clause of `clauses` that is not an element of `units` has already been used.
* `variables` is the set of all variables occurring in `clauses`.
* `used_vars`  is the set of those variables that have already been used in case distinctions.

The function `solve_recursive` solves the set $\texttt{clauses} \cup \texttt{units}$.  First, this set is saturated using unit cuts and unit subsumptions.  If the resulting set `s` contains the empty clause, it is unsatisfiable.  If it contains only unit clauses, these unit clauses form a solution.  Otherwise, we use the Jeroslow-Wang heuristic to select the most promising literal $l$ whose variable has not been used yet and perform a case distinction: First we try to solve the problem with the additional unit clause $\{l\}$.  If this fails, we try the unit clause $\{\overline{\,l\,}\}$.  The heuristic is applied to the saturated set `s`, i.e. to the clauses that remain after all unit cuts and unit subsumptions have been performed.

As `s` is saturated, all unit clauses of `s` have already been used.  Therefore, the recursive calls receive `s` together with a set that contains only the new unit clause.  The function `solve` passes all unit clauses of `clauses` to `solve_recursive`, since none of them has been used initially.

In [ ]:
let rec solve_recursive (clauses : cnf) (units : cnf) (variables : string RSet.t)
                        (used_vars : string RSet.t) : cnf =
  let s = saturate clauses units in
  let empty_clause : clause = RSet.empty in
  if RSet.mem empty_clause s then                          (* s is inconsistent *)
    RSet.singleton empty_clause
  else if RSet.for_all (fun c -> RSet.cardinal c = 1) s then  (* s is trivial *)
    s
  else
    (* use the Jeroslow-Wang heuristic to select the most promising literal l *)
    let l              = select_literal s variables used_vars in
    let next_used_vars = RSet.add (extract_variable l) used_vars in
    let result1 = solve_recursive s (RSet.singleton (RSet.singleton l))
                                  variables next_used_vars in
    if not (RSet.mem empty_clause result1) then
      result1
    else
      solve_recursive s (RSet.singleton (RSet.singleton (complement l)))
                      variables next_used_vars

In [ ]:
let solve (clauses : cnf) : cnf =
  let variables = RSet.flat_map (RSet.map extract_variable) clauses in
  let units     = RSet.filter (fun c -> RSet.cardinal c = 1) clauses in
  solve_recursive clauses units variables RSet.empty

In order to print the results, we define two functions.

The function `literal_to_str` takes one argument:
- `c` is a unit clause.

It returns a string of the form `"p ↦ True"` if `c` is the unit clause $\{p\}$ and a string of the form `"p ↦ False"` if `c` is the unit clause $\{\neg p\}$.  For the empty clause, it returns `"{}"`.

The function `to_string` takes two arguments:
- `s` is a set of clauses,
- `simplified` is the result of `solve s`.

If `simplified` contains the empty clause, it returns a string stating that `s` is unsolvable.  Here, the function `prettify` from the notebook `04-CNF.ipynb` converts `s` into a string.  Otherwise, `simplified` is a set of unit clauses and `to_string` returns a string representing the corresponding variable assignment.

In [ ]:
let literal_to_str (c : clause) : string =
  if RSet.is_empty c then
    "{}"
  else
    match first c with
    | Pos p -> p ^ " ↦ True"
    | Neg p -> p ^ " ↦ False"

let to_string (s : cnf) (simplified : cnf) : string =
  if RSet.mem RSet.empty simplified then
    prettify s ^ " is unsolvable"
  else
    "{ " ^ String.concat ", " (List.map literal_to_str (RSet.to_list simplified)) ^ " }"

In [ ]:
let c1 = RSet.of_list [Pos "r"; Pos "p"; Pos "s"]
let c2 = RSet.of_list [Pos "r"; Pos "s"]
let c3 = RSet.of_list [Pos "p"; Pos "q"; Pos "s"]
let c4 = RSet.of_list [Neg "p"; Neg "q"]
let c5 = RSet.of_list [Neg "p"; Pos "s"; Neg "r"]
let c6 = RSet.of_list [Pos "p"; Neg "q"; Pos "r"]
let c7 = RSet.of_list [Neg "r"; Neg "s"; Pos "q"]
let c8 = RSet.of_list [Neg "p"; Neg "s"]
let c9 = RSet.of_list [Pos "p"; Neg "r"; Neg "q"]
let c0 = RSet.of_list [Neg "p"; Pos "r"; Pos "q"; Neg "s"]

let s1 = RSet.of_list [c0; c1; c2; c3; c4; c5; c6; c7; c8; c9];;
print_endline (to_string s1 (solve s1))

In [ ]:
let c10 = RSet.of_list [Pos "p"; Pos "r"; Pos "q"; Neg "s"]
let s2  = RSet.of_list [c0; c1; c2; c3; c4; c5; c6; c7; c8; c9; c10];;
print_endline (to_string s2 (solve s2))

In [ ]:
let s3 = RSet.of_list [RSet.of_list [Pos "r"; Pos "p"; Pos "s"];
                       RSet.of_list [Pos "r"; Pos "s"];
                       RSet.of_list [Pos "q"; Pos "p"; Pos "s"];
                       RSet.of_list [Neg "p"; Neg "q"];
                       RSet.of_list [Neg "p"; Pos "s"; Neg "r"];
                       RSet.of_list [Pos "p"; Neg "q"; Pos "r"];
                       RSet.of_list [Neg "r"; Neg "s"; Pos "q"];
                       RSet.of_list [Pos "p"; Pos "q"; Pos "r"; Pos "s"];
                       RSet.of_list [Pos "r"; Neg "s"; Pos "q"];
                       RSet.of_list [Pos "s"; Neg "r"; Neg "q"];
                       RSet.of_list [Pos "s"; Neg "r"];
                       RSet.of_list [Pos "r"; Neg "s"]];;
print_endline (to_string s3 (solve s3))